# Sentiment Analyzer for Product Reviews

EDP Project — text preprocessing, TF-IDF, Logistic Regression, evaluation and error analysis.

In [5]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [6]:
# Load dataset
df = pd.read_csv('data/reviews.csv')
df.head()

,review,sentiment
0,This phone has an excellent camera and the bat...,positive
1,The product quality is amazing and I am very h...,positive
2,Fast delivery and the headphones sound fantastic.,positive
3,"The laptop is smooth, lightweight, and works p...",positive
4,Great value for money. The build quality feels...,positive


In [7]:
# Week 4: Text cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\\S+|www\\S+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review', 'sentiment']].head()

,review,clean_review,sentiment
0,This phone has an excellent camera and the bat...,this phone has an excellent camera and the bat...,positive
1,The product quality is amazing and I am very h...,the product quality is amazing and i am very h...,positive
2,Fast delivery and the headphones sound fantastic.,fast delivery and the headphones sound fantastic,positive
3,"The laptop is smooth, lightweight, and works p...",the laptop is smooth lightweight and works p...,positive
4,Great value for money. The build quality feels...,great value for money the build quality feels...,positive


In [8]:
# Train/test split and Week 5: Logistic Regression with TF-IDF
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'], test_size=0.25,
    random_state=42, stratify=df['sentiment']
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('classifier', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [9]:
# Weeks 6–7: Evaluation and error analysis
print('Accuracy:', accuracy_score(y_test, pred))
print('\nClassification Report:\n')
print(classification_report(y_test, pred, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=['negative', 'positive'])
print('Confusion Matrix:')
print(cm)

errors = pd.DataFrame({'review': X_test, 'actual': y_test, 'predicted': pred})
errors = errors[errors['actual'] != errors['predicted']]
print('\nMisclassified Reviews:')
display(errors)

Accuracy: 0.6666666666666666

Classification Report:

              precision    recall  f1-score   support

    negative       0.60      0.86      0.71         7
    positive       0.80      0.50      0.62         8

    accuracy                           0.67        15
   macro avg       0.70      0.68      0.66        15
weighted avg       0.71      0.67      0.66        15

Confusion Matrix:
[[6 1]
 [4 4]]

Misclassified Reviews:


,review,actual,predicted
49,setup was confusing and the instructions were ...,negative,positive
18,very satisfied with the quality and performance,positive,negative
25,the display is beautiful and the touch respons...,positive,negative
9,excellent customer service and a wonderful sho...,positive,negative
26,i am impressed by the quality at this affordab...,positive,negative


In [10]:
# Try a new review
review = input('Enter a product review: ')
print('Predicted sentiment:', model.predict([clean_text(review)])[0])


Enter a product review:  This product is excellent and the quality is amazing.


Predicted sentiment: positive


## Limitations
- The demonstration dataset is small.
- Only positive and negative sentiment are classified.
- TF-IDF + Logistic Regression may struggle with sarcasm, mixed opinions and complex context.

## Future Scope
Use a larger real-world dataset, add neutral/mixed classes, compare SVM and deploy the model as a web application.